In [ ]:
from pathlib import Path
import pandas as pd

from dask_image import imread
from spatialdata import SpatialData
from spatialdata.transformations import Identity
from spatialdata.models import PointsModel

import harpy as hp

from packaging import version
import cellpose
import torch

In [ ]:
samples = ["A1", "A2", "B2", "C2", "D1"]
file_path = "/Volumes/Intenso/spatial-transcriptomics/raw_data/"

## Loading the data

In [ ]:
def load_images_transcritps(folder: str, samples: list[str]) -> SpatialData:
    sdata = SpatialData()
    folder = Path(folder)
    for sample in samples:
        # load in the images
        matches = sorted(folder.glob(f"{sample}_DAPI.tiff"))
        p = matches[0]
        sdata = hp.im.add_image_layer(
            sdata,
            arr = imread.imread(str(p)),
            output_layer = f"{sample}_DAPI",
            transformations = {sample: Identity()},
            overwrite=True,
        )
        # loading the transcripts
        df = pd.read_csv(
            Path(folder) / f"{sample}_results.txt",
            sep = r"\s+",
            header = None,
            names = ["x", "y", "z", "gene"],
            engine = "python",
        )
        sdata.points[f"{sample}_transcripts"] = PointsModel.parse(df, coordinates={"x": "x", "y": "y"})

    return sdata

In [ ]:
sdata = load_images_transcritps(
    folder = file_path,
    samples = samples)
sdata

## Image processing

In [ ]:
# min max filtering
for sample in samples:
    sdata = hp.im.min_max_filtering(
        sdata,
        img_layer = f"{sample}_DAPI",                 # e.g. "A1_DAPI"
        output_layer = f"{sample}_min_max_filtered",  # e.g. "A1_min_max_filtered"
        size_min_max_filter = 51,
        overwrite = True,
    )

for sample in samples:
    hp.pl.plot_image(
        sdata, 
        img_layer = [f"{sample}_DAPI", f"{sample}_min_max_filtered"], 
        crd = [4000, 8000, 6000, 8000], 
        figsize = (20,20),
        to_coordinate_system = sample
    )

In [ ]:
# enhance contrast
for sample in samples:
    sdata = hp.im.enhance_contrast(
        sdata,
        img_layer = f"{sample}_min_max_filtered",
        output_layer = f"{sample}_clahe",
        contrast_clip = 20,
        chunks = 20000,
        overwrite = True
    )

# Plot the contrast enhanced image
for sample in samples:
    hp.pl.plot_image(
        sdata, 
        img_layer = [f"{sample}_min_max_filtered", f"{sample}_clahe"], 
        crd = [4000, 8000, 6000, 8000], 
        figsize = (20,20),
        to_coordinate_system = sample
    )

In [ ]:
sdata

## Cell segmentation

In [ ]:
# checking what is available on the system
cellpose_version = version.parse(cellpose.version)
if torch.backends.mps.is_available() and cellpose_version >= version.parse("4.0"):  # mps bugged in cellpose < 4.0
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}.")